In [3]:
import pandas as pd

df = pd.read_csv('C:/Users/Di/Downloads/archive/simulated_call_centre.csv')

# Посмотрим на колонки и типы данных
print(df.info())

# Посмотрим на первые 5 строк
print(df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51708 entries, 0 to 51707
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   call_id         51708 non-null  int64 
 1   date            51708 non-null  object
 2   daily_caller    51708 non-null  int64 
 3   call_started    51708 non-null  object
 4   call_answered   51708 non-null  object
 5   call_ended      51708 non-null  object
 6   wait_length     51708 non-null  int64 
 7   service_length  51708 non-null  int64 
 8   meets_standard  51708 non-null  bool  
dtypes: bool(1), int64(4), object(4)
memory usage: 3.2+ MB
None
   call_id        date  daily_caller call_started call_answered  call_ended  \
0        1  2021-01-01             1   8:00:00 AM    8:00:00 AM  8:14:22 AM   
1        2  2021-01-01             2   8:02:42 AM    8:02:42 AM  8:07:31 AM   
2        3  2021-01-01             3   8:08:24 AM    8:08:24 AM  8:10:13 AM   
3        4  2021-01-01      

In [6]:

# Шаг 1. Переводим текст в формат даты и времени

#объединим дату со временем и превратим их в полноценный формат

# 2. Переводим текстовое время в полноценный формат datetime
# Объединяем дату и время начала, ответа и конца разговора
df['start_time'] = pd.to_datetime(df['date'] + ' ' + df['call_started'])
df['answer_time'] = pd.to_datetime(df['date'] + ' ' + df['call_answered'])
df['end_time'] = pd.to_datetime(df['date'] + ' ' + df['call_ended'])

# 3. Делаем небольшую проверку данных (валидацию)
# Проверим, совпадает ли разница между ответом и началом с колонкой wait_length
calculated_wait = (df['answer_time'] - df['start_time']).dt.total_seconds().astype(int)
errors_wait = (calculated_wait != df['wait_length']).sum()
print(f"Количество нестыковок в wait_length: {errors_wait}")

# 4. Удаляем старые текстовые колонки, так как они больше не нужны
df_clean = df.drop(columns=['date', 'call_started', 'call_answered', 'call_ended'])

# 5. Смотрим на результат очистки
print(df_clean.info())
print(df_clean.head())

# 6. Сохраняем очищенный файл для базы данных и Power BI
df_clean.to_csv('call_center_clean.csv', index=False)


Количество нестыковок в wait_length: 1571
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51708 entries, 0 to 51707
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   call_id         51708 non-null  int64         
 1   daily_caller    51708 non-null  int64         
 2   wait_length     51708 non-null  int64         
 3   service_length  51708 non-null  int64         
 4   meets_standard  51708 non-null  bool          
 5   start_time      51708 non-null  datetime64[ns]
 6   answer_time     51708 non-null  datetime64[ns]
 7   end_time        51708 non-null  datetime64[ns]
dtypes: bool(1), datetime64[ns](3), int64(4)
memory usage: 2.8 MB
None
   call_id  daily_caller  wait_length  service_length  meets_standard  \
0        1             1            0             863            True   
1        2             2            0             289            True   
2        3             3            0   

In [8]:
!pip install pymysql

In [9]:
from sqlalchemy import create_engine

# Настройки подключения к локальной MySQL
# Формат: mysql+pymysql://пользователь:пароль@хост:порт/имя_базы
user = 'root'
password = ''
host = 'localhost'
database = 'call_center_analytics'

# Создаем подключение
engine = create_engine(f'mysql+pymysql://{user}:{password}@{host}/{database}')

# Записываем очищенный датафрейм в созданную таблицу call_logs
# if_exists='append' означает, что мы добавляем данные в готовую структуру таблицы
df_clean.to_sql(name='call_logs', con=engine, if_exists='append', index=False)

print("Данные успешно загружены в базу данных MySQL!")


Данные успешно загружены в базу данных MySQL!


In [11]:
# Запрос на получение всех данных из таблицы call_logs
df_from_db = pd.read_sql_query('SELECT * FROM call_logs', con=engine)

# Проверяем результат
df_from_db.head()

,call_id,daily_caller,wait_length,service_length,meets_standard,start_time,answer_time,end_time
0,1,1,0,863,1,2021-01-01 08:00:00,2021-01-01 08:00:00,2021-01-01 08:14:22
1,2,2,0,289,1,2021-01-01 08:02:42,2021-01-01 08:02:42,2021-01-01 08:07:31
2,3,3,0,108,1,2021-01-01 08:08:24,2021-01-01 08:08:24,2021-01-01 08:10:13
3,4,4,0,247,1,2021-01-01 08:09:37,2021-01-01 08:09:37,2021-01-01 08:13:45
4,5,5,0,258,1,2021-01-01 08:11:10,2021-01-01 08:11:10,2021-01-01 08:15:28


In [12]:
# Задача 1. Анализ часовой нагрузки (Ищем «часы пик»)
query_1 = """
SELECT 
    HOUR(start_time) AS call_hour,
    COUNT(call_id) AS total_calls,
    ROUND(AVG(wait_length), 2) AS avg_wait_seconds,
    ROUND(AVG(service_length), 2) AS avg_talk_seconds
FROM call_logs
GROUP BY HOUR(start_time)
ORDER BY call_hour;
"""

# Запускаем запрос через наше созданное подключение engine
df_hours = pd.read_sql_query(query_1, engine)

# Выводим результат
df_hours


,call_hour,total_calls,avg_wait_seconds,avg_talk_seconds
0,8,5434,13.75,306.26
1,9,5099,12.62,303.58
2,10,5102,13.71,299.83
3,11,5103,16.00,302.49
4,12,5236,19.50,297.29
5,13,5258,23.32,300.15
6,14,5270,17.11,304.84
7,15,5305,18.94,304.10
8,16,5225,18.07,299.73
9,17,4676,17.20,269.42


In [13]:
# Задача 2. Расчет уровня качества (SLA) по дням недели
query_2 = """
SELECT 
    DAYNAME(start_time) AS day_of_week,
    WEEKDAY(start_time) AS day_index,
    COUNT(call_id) AS total_calls,
    SUM(CASE WHEN meets_standard = 1 THEN 1 ELSE 0 END) AS successful_calls,
    ROUND(SUM(CASE WHEN meets_standard = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(call_id), 2) AS sla_percentage
FROM call_logs
GROUP BY DAYNAME(start_time), WEEKDAY(start_time)
ORDER BY day_index;
"""

df_days = pd.read_sql_query(query_2, engine)
df_days


,day_of_week,day_index,total_calls,successful_calls,sla_percentage
0,Monday,0,10244,9395.0,91.71
1,Tuesday,1,10263,9391.0,91.50
2,Wednesday,2,10344,9624.0,93.04
3,Thursday,3,10162,9402.0,92.52
4,Friday,4,10695,9669.0,90.41


In [14]:
# Задача 3. Локализация технической ошибки логирования wait_length
query_3 = """
SELECT 
    DAYNAME(start_time) AS day_of_week,
    HOUR(start_time) AS call_hour,
    COUNT(call_id) AS total_errors,
    ROUND(AVG(ABS(wait_length - TIMESTAMPDIFF(SECOND, start_time, answer_time))), 1) AS avg_error_seconds
FROM call_logs
WHERE wait_length != TIMESTAMPDIFF(SECOND, start_time, answer_time)
GROUP BY DAYNAME(start_time), HOUR(start_time), WEEKDAY(start_time)
ORDER BY total_errors DESC
LIMIT 10;
"""

df_errors = pd.read_sql_query(query_3, engine)
df_errors


,day_of_week,call_hour,total_errors,avg_error_seconds
0,Friday,13,52,1.0
1,Friday,14,47,1.0
2,Monday,13,42,1.0
3,Friday,15,42,1.0
4,Friday,12,42,1.0
5,Tuesday,8,41,1.0
6,Tuesday,15,41,1.0
7,Tuesday,13,41,1.0
8,Friday,16,40,1.0
9,Monday,15,40,1.0


In [15]:
import os
print(os.getcwd())


C:\Users\Di
